In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.preprocessing import image

from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input as mobilenet_preprocess

In [ ]:
#connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
!cp -r /content/drive/MyDrive/AML /content/Images

In [ ]:
images_root = "/content/Images"

label_map = {
    "control": 0,
    "NPM1": 1,
    "PML_RARA": 2,
    "RUNX1_RUNX1T1":3,
    "CBFB_MYH11": 4
}

data = []

for root, dirs, files in os.walk(images_root):
    for file in files:
        if file.lower().endswith(".tif"):
            img_path = os.path.join(root, file)
            parts = img_path.split(os.sep)

            patient_id = parts[-2]

            label = None
            for class_name, class_id in label_map.items():
                if class_name in parts:
                    label = class_id
                    break

            if label is not None:
                data.append([img_path, patient_id, label])

img_df = pd.DataFrame(data, columns=["img_path", "patient_id", "label"])

print("Total images:", len(img_df))
print("Total patients:", img_df["patient_id"].nunique())
print("\nClass counts:")
print(img_df["label"].value_counts().sort_index())

Total images: 81220
Total patients: 189

Class counts:
label
0    20305
1    17715
2    11585
3    14403
4    17212
Name: count, dtype: int64


In [ ]:
#Use the full image dataset for feature extraction

full_img_df = img_df.copy()

print("Using all images for feature extraction:", len(full_img_df))
print("Using all patients:", full_img_df.patient_id.nunique())


Using all images for feature extraction: 81220
Using all patients: 189


EfficientNetB0




In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)
base_model.trainable = False

IMG_SIZE = (224, 224)
BATCH_SIZE = 64

img_paths = full_img_df["img_path"].values
pids_img = full_img_df["patient_id"].values
y_img = full_img_df["label"].values

features = []

# Warm-up once
warmup_paths = img_paths[:BATCH_SIZE]
warmup_imgs = []
for path in warmup_paths:
    img = image.load_img(path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    warmup_imgs.append(img_array)

warmup_imgs = np.array(warmup_imgs, dtype=np.float32)
warmup_imgs = efficientnet_preprocess(warmup_imgs)
_ = base_model(warmup_imgs, training=False)

# Real extraction
for i in tqdm(range(0, len(img_paths), BATCH_SIZE)):
    batch_paths = img_paths[i:i + BATCH_SIZE]
    batch_imgs = []

    for path in batch_paths:
        img = image.load_img(path, target_size=IMG_SIZE)
        img_array = image.img_to_array(img)
        batch_imgs.append(img_array)

    batch_imgs = np.array(batch_imgs, dtype=np.float32)
    batch_imgs = efficientnet_preprocess(batch_imgs)

    batch_features = base_model(batch_imgs, training=False)
    features.append(batch_features)

X_img = tf.concat(features, axis=0).numpy()

print("Feature matrix shape:", X_img.shape)
print("Patient IDs shape:", pids_img.shape)
print("Labels shape:", y_img.shape)

save_path = "/content/drive/MyDrive/AML/AML_efficientnet_features.npz"
np.savez_compressed(
    save_path,
    X_img=X_img,
    y_img=y_img,
    pids_img=pids_img
)

print("Full-dataset features saved successfully!")

100%|██████████| 1270/1270 [10:07<00:00,  2.09it/s]


Feature matrix shape: (81220, 1280)
Patient IDs shape: (81220,)
Labels shape: (81220,)
Full-dataset features saved successfully!


ResNet

In [ ]:
# Load pretrained ResNet50 as feature extractor
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)
base_model.trainable = False

IMG_SIZE = (224, 224)
BATCH_SIZE = 64

img_paths = full_img_df["img_path"].values
pids_img = full_img_df["patient_id"].values
y_img = full_img_df["label"].values

features = []

# Warm-up once
warmup_paths = img_paths[:BATCH_SIZE]
warmup_imgs = []

for path in warmup_paths:
    img = image.load_img(path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    warmup_imgs.append(img_array)

warmup_imgs = np.array(warmup_imgs, dtype=np.float32)
warmup_imgs = resnet_preprocess(warmup_imgs)
_ = base_model(warmup_imgs, training=False)

# Real extraction
for i in tqdm(range(0, len(img_paths), BATCH_SIZE)):
    batch_paths = img_paths[i:i + BATCH_SIZE]
    batch_imgs = []

    for path in batch_paths:
        img = image.load_img(path, target_size=IMG_SIZE)
        img_array = image.img_to_array(img)
        batch_imgs.append(img_array)

    batch_imgs = np.array(batch_imgs, dtype=np.float32)
    batch_imgs = resnet_preprocess(batch_imgs)

    batch_features = base_model(batch_imgs, training=False)
    features.append(batch_features)

X_img = tf.concat(features, axis=0).numpy()

print("Feature matrix shape:", X_img.shape)
print("Patient IDs shape:", pids_img.shape)
print("Labels shape:", y_img.shape)

save_path = "/content/drive/MyDrive/AML/AML_resnet_features.npz"
np.savez_compressed(
    save_path,
    X_img=X_img,
    y_img=y_img,
    pids_img=pids_img
)

print("Full-dataset ResNet features saved successfully!")

100%|██████████| 1270/1270 [09:11<00:00,  2.30it/s]


Feature matrix shape: (81220, 2048)
Patient IDs shape: (81220,)
Labels shape: (81220,)
Full-dataset ResNet features saved successfully!


MobileNet

In [ ]:
# Load pretrained MobileNetV2 as feature extractor
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

IMG_SIZE = (224, 224)
BATCH_SIZE = 64

img_paths = full_img_df["img_path"].values
pids_img = full_img_df["patient_id"].values
y_img = full_img_df["label"].values

features = []

# Warm-up once
warmup_paths = img_paths[:BATCH_SIZE]
warmup_imgs = []

for path in warmup_paths:
    img = image.load_img(path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    warmup_imgs.append(img_array)

warmup_imgs = np.array(warmup_imgs, dtype=np.float32)
warmup_imgs = mobilenet_preprocess(warmup_imgs)
_ = base_model(warmup_imgs, training=False)

# Real extraction
for i in tqdm(range(0, len(img_paths), BATCH_SIZE)):
    batch_paths = img_paths[i:i + BATCH_SIZE]
    batch_imgs = []

    for path in batch_paths:
        img = image.load_img(path, target_size=IMG_SIZE)
        img_array = image.img_to_array(img)
        batch_imgs.append(img_array)

    batch_imgs = np.array(batch_imgs, dtype=np.float32)
    batch_imgs = mobilenet_preprocess(batch_imgs)

    batch_features = base_model(batch_imgs, training=False)
    features.append(batch_features)

X_img = tf.concat(features, axis=0).numpy()

print("Feature matrix shape:", X_img.shape)
print("Patient IDs shape:", pids_img.shape)
print("Labels shape:", y_img.shape)

save_path = "/content/drive/MyDrive/AML/AML_mobilenet_features.npz"
np.savez_compressed(
    save_path,
    X_img=X_img,
    y_img=y_img,
    pids_img=pids_img
)

print("Full-dataset MobileNetV2 features saved successfully!")

100%|██████████| 1270/1270 [07:10<00:00,  2.95it/s]


Feature matrix shape: (81220, 1280)
Patient IDs shape: (81220,)
Labels shape: (81220,)
Full-dataset MobileNetV2 features saved successfully!
